# FD solver vs Abaqus FEA

Digambar's Abaqus run (see `Digambar Data/DIGAMBAR_DATA_INVENTORY.md`) is the first real
reference for `fd_solver.ipynb`. This notebook puts the two side by side at one interface
and works out why they differ.

**They are not expected to match.**

- **The FD solver** is 1D (height only). It puts down one layer that is already hot, on top
  of a uniform block.
- **The FEA** is 3D. It heats each new cell up over the time it takes to print it, and it builds
  the wall layer by layer on a coarse mesh.

The goal is to measure the gap and work out where it comes from.

Steps:
1. Pick an FEA interface and the time window to compare.
2. Set the FD solver's parameters to the FEA's values.
3. Compare the two curves: peak, time to cool to 80 °C, and the shape of the decay.
4. Add the FEA's modelling choices to the FD solver one at a time, to see which ones explain the gap.

Needs `fea_loader.py` (repo root) and the `Digambar Data/` folder.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from fea_loader import load_fea_thermal, load_activation_times

# Values from the FEA deck (gcode_run/wall_thermal.inp and run_summary.json)
rho, cp = 1240.0, 1800.0        # kg/m^3, J/(kg K): same in the FD solver and the FEA
k_fea   = 0.18                  # W/(m K)
h_fea   = 15.0                  # W/(m^2 K): still air; the FEA ignores the part fan
T_inf   = 25.0                  # degC, ambient
T_bed   = 60.0                  # degC
T_dep   = 220.0 - 15.0          # degC: nozzle setpoint minus the deck's assumed 15 degC drop
LH      = 0.2e-3                # m, layer height
width   = 0.516e-3              # m, wall (road) width
cell_mm = 70.684 / 10           # mm, length of one FEA cell along the wall
speed   = 10.6                  # mm/s, wall print speed from layer 3 up
t_cell  = cell_mm / speed       # s, time to print one cell = heating time of a new FEA cell

print(f"one FEA cell is {cell_mm:.2f} mm long and takes {t_cell:.3f} s to print")

## 1. The FEA interface

We use **interface 64**, at **x = 38.88 mm**.
- Interface 64 is the bond between layers 63 and 64, 12.6 mm above the bed.
- x = 38.88 mm is the middle of the wall, far from the bed and from the wall ends.

How to read the history:
- Each interface history starts (t = 0) when the cell above it is switched on.
- About 7.4 s later the next layer is printed on top, and the interface warms up again.
- The single-layer FD solver can't represent that, so the comparison stops at the point where
  the FEA curve starts rising again.

In [ ]:
fea = load_fea_thermal()
act = load_activation_times()
print(fea)

I, X = 64, 38.8762                       # interface id, position along the wall (mm)
t_fea, T_fea = fea.get(I, X)

t_next = act[(I + 1, X)] - act[(I, X)]   # when the next layer is printed above this point
search = (t_fea > 3.0) & (t_fea < t_next)
t_win = t_fea[np.flatnonzero(search)[np.argmin(T_fea[search])]]   # FEA starts warming again

print(f"interface {I} is {(I - 1) * 0.2:.1f} mm above the bed")
print(f"next layer printed at t = {t_next:.2f} s; FEA starts warming up at t = {t_win:.2f} s")
print(f"comparison window: 0 to {t_win:.2f} s")

In [ ]:
arrivals = [act[(j, X)] - act[(I, X)] for j in range(I + 1, I + 4)]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(t_fea, T_fea, color="tab:blue", label=f"FEA, interface {I}, x = {X:.2f} mm")
ax.axvspan(0, t_win, color="tab:blue", alpha=0.08, label="comparison window")
for n, ta in enumerate(arrivals):
    ax.axvline(ta, color="grey", ls="--", lw=1, label="next layers printed above" if n == 0 else None)
ax.axhline(60, color="grey", ls=":", lw=1)
ax.text(24.9, 61, "$T_g$", ha="right", va="bottom", color="grey")
ax.set_xlabel("time since this interface's upper cell was switched on (s)")
ax.set_ylabel("interface temperature (degC)")
ax.set_title("Full 25 s FEA history: each new layer reheats the interface")
ax.legend(fontsize=8, loc="upper right")
plt.tight_layout(); plt.show()

### A mesh effect to know about

**The FEA mesh is very coarse along the wall:** 10 cells per layer, each 7.07 mm long.

**What the "interface temperature" is.** It's the average of the 4 corner nodes of the face
between two cells. Those corner nodes are shared with the neighbouring cells along the wall,
so each reading is also heated by:
- **the cell printed just before it.** This pre-heats it, so it's already warm at t = 0.
- **the cell printed just after it.** This delays the peak until that cell is finished, at
  2 × 0.667 = 1.33 s.

The table below shows both effects:
- The first cell (x = 3.53 mm) has no cell before it, and it starts about 16 °C cooler than
  the others.
- The last cell (x = 67.15 mm) has no cell after it, and it peaks at 0.67 s instead of 1.33 s.

In [ ]:
print(" x (mm)   T at t=0 (degC)   peak (degC)   time of peak (s)")
for x in fea.positions(I):
    t, T = fea.get(I, x)
    print(f"{x:7.2f} {T[0]:17.1f} {T.max():13.1f} {t[T.argmax()]:18.2f}")

## 2. Matching the parameters

| Parameter | `fd_solver.ipynb` | FEA | Used here |
|---|---|---|---|
| k (W/m·K) | 0.13 | 0.18 | **0.18** |
| h, top surface (W/m²K) | 50 | 15 | **15** |
| ρ, c_p | 1240, 1800 | 1240, 1800 | same |
| Ambient | 25 °C | 25 °C | 25 °C |
| Bed | fixed 60 °C | 60 °C through h = 500 | fixed 60 °C (see below) |
| Layer height | 0.2 mm | 0.2 mm | 0.2 mm |
| New material | placed at 210 °C | heated to 205 °C (220 setpoint − 15 °C drop) | **placed at 205 °C** |
| Material underneath | uniform 40 °C | about 79 °C (heat builds up layer after layer) | **78.8 °C, read from the FEA** |

**Bed.** It doesn't matter at this interface. It's 12.6 mm below in the FEA and 4.8 mm below in
the FD domain, while heat only spreads about √(α·t) ≈ 0.75 mm in 7 s.

**Material underneath.** This row matters most. In the FEA the wall underneath is still warm
from the layers before. The FD solver starts it at 40 °C.
- The value comes from the history of **interface 63**, 0.2 mm lower, at the moment layer 64
  arrives.
- Interface 64's own t = 0 value (88.4 °C) isn't used, because it includes the pre-heat from
  the previous cell described above.
- Interfaces 62 and 61 give the temperature further down. This is used in section 5.

In [ ]:
# Temperature under the interface when layer 64 arrives, read from the interfaces below it.
depth_mm, T_below = [], []
for j in (I - 1, I - 2, I - 3):
    elapsed = act[(I, X)] - act[(j, X)]            # time since interface j was deposited
    t_j, T_j = fea.get(j, X)
    depth_mm.append((I - j) * 0.2)
    T_below.append(float(np.interp(elapsed, t_j, T_j)))
    print(f"interface {j}: {depth_mm[-1]:.1f} mm below, deposited {elapsed:5.2f} s earlier "
          f"-> {T_below[-1]:.1f} degC")

T_sub = T_below[0]          # one substrate temperature for the FD solver


def substrate_profile(depth_m):
    """FEA temperature vs depth below the interface (m). Flat beyond the known points."""
    return np.interp(np.asarray(depth_m) * 1e3, depth_mm, T_below)

## 3. The FD solver

This is the same solver as in `fd_solver.ipynb`, with three optional additions that are **off by
default**. They're used in section 5.

- **`T_prev` can be a function of depth below the interface**, instead of one number.
- **`side_width`** adds heat loss through the two side faces of a wall that thin. The 1D solver
  otherwise treats the wall as infinitely wide.
- **`deposition="flux"`** puts the new layer down the way the FEA does: it starts at 25 °C and a
  heat source raises it to `T_nozzle` over `t_heat` seconds. The interface node gets half the
  source, because half of its cell is new material.

The defaults are `fd_solver.ipynb`'s values. The cell after the solver checks that they reproduce
that notebook's output.

In [ ]:
def solve_wall(k=0.13, rho=rho, cp=cp, H=5e-3, LH=0.2e-3, nz=251,
               T_nozzle=210.0, T_prev=40.0, T_bed=60.0, T_inf=25.0,
               h=50.0, t_end=20.0, n_save=400, safety=0.4,
               side_width=None, deposition="instant", t_heat=None):
    """1D transient conduction through the build direction (fd_solver.ipynb + options).

    Returns z (nz,), ts (n_t,), Ts (n_t, nz) in m, s, degC.
    """
    alpha = k / (rho * cp)
    dz = H / (nz - 1)
    dt = safety * dz**2 / alpha              # stability-limited step (Fo = safety)
    nt = int(t_end / dt)
    z = np.linspace(0.0, H, nz)

    T_old = T_prev((H - LH) - z) if callable(T_prev) else T_prev
    i_if = np.argmin(np.abs(z - (H - LH)))   # interface node
    source = np.zeros(nz)                    # degC/s while the new layer is being heated
    if deposition == "instant":              # original FD: new layer starts hot
        T = np.where(z >= H - LH, T_nozzle, T_old)
    elif deposition == "flux":               # FEA style: new layer starts at T_inf and is heated
        new = np.arange(nz) > i_if
        T = np.where(new, T_inf, T_old)
        source[new] = (T_nozzle - T_inf) / t_heat
        source[i_if] = 0.5 * (T_nozzle - T_inf) / t_heat
    else:
        raise ValueError("deposition must be 'instant' or 'flux'")
    T = np.array(T, dtype=float)
    T[0] = T_bed
    side = 0.0 if side_width is None else 2 * h / (rho * cp * side_width)   # 1/s

    save_every = max(1, nt // n_save)
    ts, Ts = [], []
    for n in range(nt + 1):
        if n % save_every == 0:
            ts.append(n * dt)
            Ts.append(T.copy())

        Tn = T
        Tnew = Tn.copy()
        # interior nodes
        Tnew[1:-1] = Tn[1:-1] + alpha*dt/dz**2 * (Tn[2:] - 2*Tn[1:-1] + Tn[:-2])
        # bottom: fixed bed temperature
        Tnew[0] = T_bed
        # top: convection to ambient (ghost node)
        Tnew[-1] = Tn[-1] + 2*alpha*dt/dz**2 * (
            Tn[-2] - Tn[-1] - dz*h/k * (Tn[-1] - T_inf))
        # optional additions
        if deposition == "flux" and n * dt < t_heat:
            Tnew[1:] += source[1:] * dt
        if side:
            Tnew[1:] -= side * dt * (Tn[1:] - T_inf)

        T = Tnew

    return z, np.array(ts), np.array(Ts)


def interface_series(z, Ts, H=5e-3, LH=0.2e-3):
    """Temperature history of the interface node (bottom of the new layer)."""
    return Ts[:, np.argmin(np.abs(z - (H - LH)))]


def time_above(ts, T_series, T_thresh):
    """Seconds the series spends above a threshold."""
    return float(np.trapezoid((T_series > T_thresh).astype(float), ts))

In [ ]:
# Check: with fd_solver.ipynb's settings this reproduces that notebook's printed output.
z, ts, Ts = solve_wall()
T_if = interface_series(z, Ts)
T_check = np.interp([0, 0.5, 1, 2, 5], ts, T_if)
print("interface T at 0, 0.5, 1, 2, 5 s:", T_check.round(1), "   fd_solver.ipynb: [210. 117. 102. 85.1 65.6]")
print(f"time above Tg = {time_above(ts, T_if, 60.0):.2f} s                     fd_solver.ipynb: 6.95 s")
assert np.allclose(T_check, [210.0, 117.0, 102.0, 85.1, 65.6], atol=0.06)

## 4. FD vs FEA

Three curves over the comparison window:
- **FEA:** interface 64.
- **FD matched:** the parameters from section 2.
- **FD as in `fd_solver.ipynb`:** unchanged, for reference.

The table columns:
- `peak`, `t_peak`: highest temperature and when it happens. Values before t = 0.05 s are
  skipped. The FD interface node *starts* at the new layer's temperature and drops to about the
  average of the two layers within a few milliseconds. That start is a modelling shortcut, not
  a real peak.
- `T_1s`, `T_2s`, `T_5s`, `T_end`: temperature at 1, 2 and 5 s, and at the end of the window.
- `t_80`: time from deposition until the interface first reaches 80 °C.
- `t_half`: time from the peak until the curve has fallen halfway from its peak to `T_end`.
  This describes the shape of the decay.

In [ ]:
matched = dict(k=k_fea, h=h_fea, T_nozzle=T_dep, T_bed=T_bed, T_inf=T_inf,
               t_end=t_win, n_save=2000)

z, ts_fd, Ts_fd = solve_wall(T_prev=T_sub, **matched)
T_fd = interface_series(z, Ts_fd)

z0, ts_0, Ts_0 = solve_wall(t_end=t_win, n_save=2000)      # fd_solver.ipynb settings
T_fd0 = interface_series(z0, Ts_0)

in_win = t_fea <= t_win
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(t_fea[in_win], T_fea[in_win], color="tab:blue", lw=2.5, label="FEA (interface 64)")
ax.plot(ts_fd, T_fd, color="tab:red", label="FD matched to the FEA")
ax.plot(ts_0, T_fd0, color="grey", ls="--", lw=1, label="FD as in fd_solver.ipynb")
ax.axhline(80, color="black", ls=":", lw=0.8); ax.text(t_win, 81, "80 degC", ha="right", fontsize=8)
ax.axhline(60, color="grey", ls=":", lw=0.8); ax.text(t_win, 61, "$T_g$", ha="right", color="grey")
ax.set_ylim(40, 215)
ax.set_xlabel("time since deposition (s)"); ax.set_ylabel("interface temperature (degC)")
ax.set_title("Interface temperature: FD solver vs FEA")
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
def summarize(t, T, t_end):
    """Peak, temperatures at fixed times and cooling times over 0..t_end."""
    keep = t <= t_end + 1e-9
    t, T = t[keep], T[keep]
    late = t >= 0.05                          # skip the FD's instantaneous start
    i_pk = np.flatnonzero(late)[np.argmax(T[late])]
    t_pk, T_pk = t[i_pk], T[i_pk]
    after = t > t_pk
    hit80 = np.flatnonzero(after & (T <= 80.0))
    half = T_pk - 0.5 * (T_pk - T[-1])
    hit_half = np.flatnonzero(after & (T <= half))
    return {"peak": T_pk, "t_peak": t_pk,
            "T_1s": np.interp(1, t, T), "T_2s": np.interp(2, t, T), "T_5s": np.interp(5, t, T),
            "T_end": T[-1],
            "t_80": t[hit80[0]] if hit80.size else np.nan,
            "t_half": t[hit_half[0]] - t_pk if hit_half.size else np.nan}


def print_table(rows):
    cols = ["peak", "t_peak", "T_1s", "T_2s", "T_5s", "T_end", "t_80", "t_half"]
    print(f"{'':34s}" + "".join(f"{c:>8s}" for c in cols))
    for name, s in rows:
        print(f"{name:34s}" + "".join(f"{s[c]:8.1f}" if c != "t_peak" and np.isfinite(s[c])
                                      else f"{s[c]:8.2f}" if np.isfinite(s[c])
                                      else f"{'never':>8s}" for c in cols))


print_table([("FEA (interface 64)", summarize(t_fea, T_fea, t_win)),
             ("FD matched", summarize(ts_fd, T_fd, t_win)),
             ("FD as in fd_solver.ipynb", summarize(ts_0, T_fd0, t_win))])

In [ ]:
# Shape of the decay: each curve scaled to run from 1 at its peak to 0 at the end of the window.
fig, ax = plt.subplots(figsize=(7, 4))
for (t, T, label, color) in [(t_fea, T_fea, "FEA", "tab:blue"), (ts_fd, T_fd, "FD matched", "tab:red")]:
    s = summarize(t, T, t_win)
    m = (t >= s["t_peak"]) & (t <= t_win)
    ax.plot(t[m] - s["t_peak"], (T[m] - s["T_end"]) / (s["peak"] - s["T_end"]), color=color,
            label=f"{label}: half of the drop after {s['t_half']:.2f} s")
ax.axhline(0.5, color="grey", ls=":", lw=0.8)
ax.set_xlabel("time since peak (s)"); ax.set_ylabel("(T - T_end) / (peak - T_end)")
ax.set_title("Decay shape, normalised")
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 5. Where does the gap come from?

Each run below starts from **FD matched** and adds one thing the FEA does. The last two runs add
several together.

**Nothing here is fitted.** Every number comes from the FEA deck or from the FEA's own output.

| Run | What changes |
|---|---|
| `+ substrate profile` | The material underneath gets the FEA's temperature at each depth (78.8 / 75.2 / 72.9 °C at 0.2 / 0.4 / 0.6 mm), not one value. |
| `+ side cooling` | Heat loss through the two side faces of the 0.516 mm wall (h = 15). |
| `+ heat-source deposition` | The new layer starts at 25 °C and is heated to 205 °C over 0.667 s, like the FEA. |
| `+ one node per layer` | FD grid spacing of 0.2 mm, like the FEA's one element per layer. A smaller `safety` keeps the time step at about 0.01 s, so the 0.667 s heating is resolved. |

In [ ]:
variants = {
    "FD matched":                   dict(T_prev=T_sub),
    "+ substrate profile":          dict(T_prev=substrate_profile),
    "+ side cooling":               dict(T_prev=T_sub, side_width=width),
    "+ heat-source deposition":     dict(T_prev=T_sub, deposition="flux", t_heat=t_cell),
    "+ one node per layer":         dict(T_prev=T_sub, nz=26, safety=0.02),
    "profile + side + heat source": dict(T_prev=substrate_profile, side_width=width,
                                         deposition="flux", t_heat=t_cell),
    "  ... + one node per layer":   dict(T_prev=substrate_profile, side_width=width,
                                         deposition="flux", t_heat=t_cell, nz=26, safety=0.02),
}

runs = {}
for name, extra in variants.items():
    z_v, ts_v, Ts_v = solve_wall(**matched, **extra)
    runs[name] = (ts_v, interface_series(z_v, Ts_v))

print_table([("FEA (interface 64)", summarize(t_fea, T_fea, t_win))] +
            [(name, summarize(t, T, t_win)) for name, (t, T) in runs.items()])

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(t_fea[in_win], T_fea[in_win], color="tab:blue", lw=2.5, label="FEA (interface 64)")
for name, color in [("FD matched", "tab:red"),
                    ("+ side cooling", "tab:orange"),
                    ("profile + side + heat source", "tab:green"),
                    ("  ... + one node per layer", "tab:purple")]:
    t, T = runs[name]
    ax.plot(t, T, color=color, lw=1.2, label=name.strip())
ax.axhline(80, color="black", ls=":", lw=0.8)
ax.set_ylim(40, 215)
ax.set_xlabel("time since deposition (s)"); ax.set_ylabel("interface temperature (degC)")
ax.set_title("Adding the FEA's modelling choices to the FD solver")
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 6. Findings

Interface 64, x = 38.88 mm, over the first 6.95 s (before the next layer starts reheating it):

| | FEA | FD matched |
|---|---|---|
| Peak | 116.8 °C at 1.33 s | about 142 °C straight after deposition (the average of 205 and 78.7 °C); 147.3 °C at 0.05 s |
| T at 2 s | 105.3 °C | 110.6 °C |
| T at 5 s | 82.9 °C | 97.7 °C |
| Time to cool to 80 °C | 5.7 s | not reached (93.8 °C at 6.95 s) |
| Decay shape | falls steadily and keeps falling; half of its drop takes 1.5 s | falls fast, then levels off; half of its drop takes 1.0 s |

**1. The temperature of the material underneath matters most.**
- In the FEA the wall under the new layer is about 79 °C when it arrives, because heat has built up over
  63 earlier layers.
- The FD's 5 s temperature is 67.1 °C with 40 °C underneath and 97.7 °C with 78.7 °C underneath.
- That 30 °C shift is bigger than every other difference combined. A single-layer model has to be told what
  temperature the wall is at when the layer arrives, and it can't work that out on its own.

**2. Side cooling explains most of the remaining late-time gap.**
- The 0.516 mm wall loses heat through both faces, and a 1D model has no side faces.
- Adding them lowers the FD's 5 s temperature by 8.9 °C, and its end temperature by 11.4 °C.

**3. The cooler material deeper down adds about 3 °C** (97.7 → 94.7 °C at 5 s).

**4. Heat-source deposition mainly changes the first second.**
- The FD interface no longer jumps instantly to about 142 °C. It rises to a peak at 0.67 s, when the cell
  finishes heating, like the FEA's first rise.
- By 5 s the difference is under 1 °C.
- The brief dip at the very start of that run comes from the new material starting at 25 °C right next to
  the interface. The FEA's coarse mesh hides most of that dip.

**5. With items 1–4 included, the FD is within 2.5–4 °C of the FEA from 2 s on:**
107.8 vs 105.3 °C at 2 s, 86.9 vs 82.9 °C at 5 s, and 80.1 vs 76.1 °C at 6.95 s.

**6. What's left is the first ~1.5 s.** The FD peaks at 138.0 °C at 0.67 s; the FEA peaks at 116.8 °C at 1.33 s.
- This matches the mesh effect from section 1. The FEA reading shares corner nodes with the neighbouring
  7 mm cells, so its heating is spread over three cells' print times: it's pre-heated before t = 0, and it
  gets a second bump at 1.33 s.
- A 1D model can't reproduce that. A finer FEA mesh along the wall would probably reduce it.

**7. Using one node per layer, like the FEA's resolution in height, makes the FD hotter, not cooler**
(peak 146.5 vs 138.0 °C, and 90.6 vs 86.9 °C at 5 s). So the FEA's one-element-per-layer mesh doesn't explain
its lower peak.

### What this means

- **Once the inputs match, the FD solver and the FEA agree to within a few degrees after the first ~2 s.**
  The remaining differences come from three things: the geometry (side cooling), the FEA's way of depositing
  material, and the FEA's coarse mesh.
- **For a thin wall, a 1D model has to get two things right:** the temperature of the wall underneath, and
  the heat lost through the sides. Adding a side-cooling term to `fd_solver.ipynb` would be a small change,
  but whether to make it is a separate decision.
- **Time above T_g can't be compared yet.** Neither curve drops to 60 °C in this window, and later layers
  reheat the FEA interface before it gets there. That needs the longer extraction from the `.odb`. Abaqus
  isn't installed on this machine, so Digambar has to run it.
- **Agreement with the FEA isn't agreement with reality.** The FEA itself is uncalibrated: h = 15 W/m²K
  ignores the 100% part fan, the properties are constant, and the mesh is coarse.
- **Limits of this comparison:**
  - It uses one interface at one position.
  - The FEA temperature underneath is only known to 0.6 mm depth; it's held constant below that.
  - The FEA's way of re-activating cells is only approximated in the FD.